# 01 — Data Exploration

Explore raw Norwegian IoT battery sensor data: quality, voltage decay curves, seasonal temperature, and lifetime stats.

In [1]:
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns
from pathlib import Path

sns.set_theme(style='whitegrid', palette='muted')
BASE = Path('..')
RAW  = BASE / 'data' / 'raw'
RES  = BASE / 'results'
RES.mkdir(parents=True, exist_ok=True)
print('Paths OK. Results dir:', RES.resolve())

Paths OK. Results dir: /home/user/battery_swap_ai_2026/results


## 1. Load Data

In [2]:
df        = pd.read_csv(RAW / 'sensor_readings.csv', parse_dates=['timestamp'])
buildings = pd.read_csv(RAW / 'buildings.csv')
travel    = pd.read_csv(RAW / 'travel_times.csv')
print(f'sensor_readings : {df.shape[0]:,} rows x {df.shape[1]} cols')
print(f'buildings       : {buildings.shape[0]} rows')
print(f'travel_times    : {travel.shape[0]} rows')
print(df.dtypes)

sensor_readings : 9,205 rows x 6 cols
buildings       : 20 rows
travel_times    : 400 rows
sensor_id                      str
building_id                    str
timestamp           datetime64[us]
voltage                    float64
temperature                float64
end_of_life_date               str
dtype: object


## 2. Data Quality Report

In [3]:
total = len(df)
print('=' * 50)
print('          DATA QUALITY REPORT')
print('=' * 50)
for col in df.columns:
    n_missing = df[col].isna().sum()
    pct = n_missing / total * 100
    print(f'  {col:<24}: {n_missing:>5} missing  ({pct:5.1f}%)')
print('=' * 50)
print(f'  Total rows            : {total:,}')
print(f'  Unique sensors        : {df.sensor_id.nunique()}')
print(f'  Unique buildings      : {df.building_id.nunique()}')
print(f'  Date range            : {df.timestamp.min().date()} to {df.timestamp.max().date()}')
print(f'  Voltage range         : {df.voltage.min():.3f}V - {df.voltage.max():.3f}V')
print(f'  Temperature range     : {df.temperature.min():.1f}C - {df.temperature.max():.1f}C')
print('=' * 50)

          DATA QUALITY REPORT
  sensor_id               :     0 missing  (  0.0%)
  building_id             :     0 missing  (  0.0%)
  timestamp               :     0 missing  (  0.0%)
  voltage                 :     0 missing  (  0.0%)
  temperature             :     0 missing  (  0.0%)
  end_of_life_date        :  9190 missing  ( 99.8%)
  Total rows            : 9,205
  Unique sensors        : 50
  Unique buildings      : 20
  Date range            : 2025-01-02 to 2026-05-03
  Voltage range         : 2.481V - 4.221V
  Temperature range     : -15.6C - 26.6C


## 3. Battery Lifetime Stats

In [4]:
# Get death dates from rows where end_of_life_date is set
dead_sensor_ids = df[df['end_of_life_date'].notna()]['sensor_id'].unique()
death_dates = (df[df['end_of_life_date'].notna()]
               .groupby('sensor_id')['end_of_life_date'].first()
               .reset_index().rename(columns={'end_of_life_date': 'death_date'}))
death_dates['death_date'] = pd.to_datetime(death_dates['death_date'])
# First reading from ALL rows (including the pre-death alive rows) for each dead sensor
first_readings = (df[df['sensor_id'].isin(dead_sensor_ids)]
                  .groupby('sensor_id')['timestamp'].min()
                  .reset_index().rename(columns={'timestamp': 'first_reading'}))
dead_sensors = first_readings.merge(death_dates, on='sensor_id')
dead_sensors['lifetime_days'] = (dead_sensors['death_date'] - dead_sensors['first_reading']).dt.days

lt = dead_sensors['lifetime_days']
print('=' * 50)
print('   BATTERY LIFETIME STATS (dead sensors only)')
print('=' * 50)
print(f'  Dead sensors count    : {len(dead_sensors)}')
print(f'  Mean lifetime (days)  : {lt.mean():.1f}')
print(f'  Median lifetime (days): {lt.median():.1f}')
print(f'  Min lifetime (days)   : {lt.min()}')
print(f'  Max lifetime (days)   : {lt.max()}')
print('=' * 50)
print(dead_sensors[['sensor_id','lifetime_days','death_date']].sort_values('lifetime_days').to_string(index=False))


   BATTERY LIFETIME STATS (dead sensors only)
  Dead sensors count    : 15
  Mean lifetime (days)  : 165.9
  Median lifetime (days): 167.0
  Min lifetime (days)   : 58
  Max lifetime (days)   : 253
sensor_id  lifetime_days death_date
  SEN_015             58 2025-06-29
  SEN_008             61 2025-03-05
  SEN_013             90 2025-05-07
  SEN_006            122 2025-06-26
  SEN_007            128 2025-07-09
  SEN_009            130 2025-06-02
  SEN_014            132 2025-06-30
  SEN_002            167 2025-08-07
  SEN_012            183 2025-10-25
  SEN_005            215 2025-12-09
  SEN_003            223 2025-12-27
  SEN_010            233 2025-12-02
  SEN_011            246 2025-11-07
  SEN_004            248 2025-10-06
  SEN_001            253 2026-03-05


## 4. Plot: Voltage Decay Curves

In [5]:
dead_ids  = dead_sensors.sort_values('lifetime_days')['sensor_id'].tolist()[:3]
alive_ids = df[df['end_of_life_date'].isna()].groupby('sensor_id').size().nlargest(3).index.tolist()
sample_ids = dead_ids + alive_ids

fig, axes = plt.subplots(2, 3, figsize=(15, 8), sharey=True)
axes = axes.flatten()

for i, sid in enumerate(sample_ids):
    sdf = df[df['sensor_id'] == sid].sort_values('timestamp')
    is_dead = sid in dead_ids
    color = '#e74c3c' if is_dead else '#27ae60'
    status_label = 'DEAD' if is_dead else 'ALIVE'
    axes[i].plot(sdf['timestamp'], sdf['voltage'], color=color, linewidth=1.4, alpha=0.85)
    axes[i].axhline(2.5, color='#2c3e50', linestyle='--', linewidth=0.9, alpha=0.6, label='Dead threshold 2.5V')
    axes[i].set_title(f'{sid}  [{status_label}]', fontsize=10, fontweight='bold', color=color)
    axes[i].set_ylabel('Voltage (V)', fontsize=8)
    axes[i].set_ylim(1.5, 4.6)
    axes[i].xaxis.set_major_formatter(mdates.DateFormatter("%b'%y"))
    axes[i].xaxis.set_major_locator(mdates.MonthLocator(interval=2))
    plt.setp(axes[i].xaxis.get_majorticklabels(), rotation=30, fontsize=7)
    if i == 0:
        axes[i].legend(fontsize=7)

fig.suptitle('Voltage Decay Curves — Sample Sensors  (Red=Dead  Green=Alive)', fontsize=12, fontweight='bold')
plt.tight_layout()
out = RES / 'voltage_curves_sample.png'
plt.savefig(out, dpi=150, bbox_inches='tight')
plt.close()
print('Saved:', out)

Saved: ../results/voltage_curves_sample.png


## 5. Plot: Seasonal Temperature

In [6]:
df['month'] = df['timestamp'].dt.month
month_names = ['Jan','Feb','Mar','Apr','May','Jun','Jul','Aug','Sep','Oct','Nov','Dec']
monthly = df.groupby('month')['temperature'].agg(['mean','std','min','max']).reset_index()

fig, ax = plt.subplots(figsize=(12, 5))
ax.fill_between(monthly['month'],
                monthly['mean'] - monthly['std'],
                monthly['mean'] + monthly['std'],
                alpha=0.25, color='#2980b9', label='Mean ± 1 std')
ax.fill_between(monthly['month'], monthly['min'], monthly['max'],
                alpha=0.08, color='#2980b9', label='Full range')
ax.plot(monthly['month'], monthly['mean'], 'o-', color='#1a5276',
        linewidth=2.5, markersize=8, label='Mean temperature')
ax.axhline(0, color='navy', linestyle=':', linewidth=1.1, alpha=0.6, label='0°C')
ax.axhspan(-20, 0, alpha=0.04, color='#2980b9')
ax.set_xticks(range(1, 13))
ax.set_xticklabels(month_names)
ax.set_xlabel('Month', fontsize=11)
ax.set_ylabel('Temperature (°C)', fontsize=11)
ax.set_title('Norwegian Seasonal Temperature Distribution (All Sensor Readings)', fontsize=13, fontweight='bold')
ax.legend(fontsize=9)
plt.tight_layout()
out = RES / 'temperature_seasonal.png'
plt.savefig(out, dpi=150, bbox_inches='tight')
plt.close()
print('Saved:', out)

Saved: ../results/temperature_seasonal.png


## 6. Plot: Voltage vs Temperature Scatter

In [7]:
sample = df.sample(n=min(4000, len(df)), random_state=42).copy()
sample['status'] = sample['end_of_life_date'].notna().map({True: 'Dead sensor', False: 'Alive sensor'})

fig, ax = plt.subplots(figsize=(10, 6))
colors_map = {'Alive sensor': '#27ae60', 'Dead sensor': '#e74c3c'}
for status, grp in sample.groupby('status'):
    ax.scatter(grp['temperature'], grp['voltage'],
               c=colors_map[status], alpha=0.30, s=10, label=status, edgecolors='none')

z = np.polyfit(sample['temperature'], sample['voltage'], 1)
xline = np.linspace(sample['temperature'].min(), sample['temperature'].max(), 200)
ax.plot(xline, np.poly1d(z)(xline), 'k--', linewidth=1.8, alpha=0.65,
        label=f'OLS trend  (slope = {z[0]:.4f} V/°C)')
ax.axhline(2.5, color='#7f8c8d', linestyle=':', linewidth=1.1, alpha=0.7, label='Dead threshold 2.5V')
ax.axvline(0,   color='#2980b9', linestyle=':', linewidth=1.0, alpha=0.5, label='0°C')

corr = sample[['temperature','voltage']].corr().iloc[0,1]
ax.set_xlabel('Temperature (°C)', fontsize=11)
ax.set_ylabel('Voltage (V)', fontsize=11)
ax.set_title(f'Voltage vs Temperature  —  Pearson r = {corr:.3f}', fontsize=13, fontweight='bold')
ax.legend(fontsize=9, markerscale=2.5)
plt.tight_layout()
out = RES / 'voltage_temperature_scatter.png'
plt.savefig(out, dpi=150, bbox_inches='tight')
plt.close()
print('Saved:', out)

Saved: ../results/voltage_temperature_scatter.png


## 7. File Check

In [8]:
files = ['voltage_curves_sample.png', 'temperature_seasonal.png', 'voltage_temperature_scatter.png']
print('\nOutput file check:')
for fname in files:
    p = RES / fname
    ok = p.exists()
    kb = p.stat().st_size // 1024 if ok else 0
    print(f'  {"OK" if ok else "MISSING"}  {fname}  ({kb} KB)')


Output file check:
  OK  voltage_curves_sample.png  (183 KB)
  OK  temperature_seasonal.png  (135 KB)
  OK  voltage_temperature_scatter.png  (429 KB)
